# Name: [Student Name]
# Student Number: [stXXXXXXXX]
# Module: PDAN8411 - Programming for Data Analytics
# Assessment: POE Part 2 (Classification)

---
## 1. Introduction & Justification of Suitability

**Task:** Answer the following in your own words.

### 1a. Why is Classification the right approach?
Explain why classification (and not regression) is appropriate here. Think about the nature of your target variable — what type of output does classification produce, and how does that match the problem you are solving?

> *Example starting point:* "Classification is appropriate for this dataset because the target variable is categorical (e.g., Malignant or Benign). Unlike regression which predicts a continuous value, a classification algorithm assigns each observation to a discrete class. In the context of cancer diagnosis, this means the model will predict whether a tumour is malignant or benign — a binary decision with direct clinical relevance."

### 1b. Dataset Justification & Quality
You must address **all four** of the following points:

1. **Source credibility** — Where does the dataset come from? Is it a well-established benchmark dataset (e.g., the UCI Wisconsin Breast Cancer Dataset)? Was it published by a reputable institution?
2. **Kaggle quality score** — If sourced from Kaggle, what is its usability score? A score of 8.0+ generally indicates a well-curated dataset. State the score and what it implies.
3. **Dataset suitability for classification** — What is the target variable? Is it clearly defined and balanced? How many features are available, and are they relevant to the prediction task?
4. **Known pitfalls & your mitigation plan** — Every real-world dataset has pitfalls. Identify at least two (e.g., class imbalance, missing values, correlated features, outliers) and briefly state how you plan to address each.

> *Example pitfall table:*
>
> | Pitfall | Mitigation Strategy |
> |---|---|
> | Class imbalance (more benign than malignant cases) | Use stratified train/test split; report precision & recall, not just accuracy |
> | Highly correlated features (multicollinearity) | Visualise with a heatmap; consider dropping redundant features or using feature selection |

---
## 2. Analysis Plan

**Requirement:** Before writing any code, plan your entire analysis. This section is assessed — marks will be deducted for vague or wordy plans. Be succinct and structured.

Your plan must cover all five areas below. You may use a table, bullet points, or an infographic — whatever communicates your intent clearly.

### Plan Summary

| Step | What I Will Do | Tools / Techniques |
|---|---|---|
| **a. EDA** | Check for nulls, duplicates; visualise feature distributions and class balance; examine correlations | `df.isnull()`, `sns.heatmap`, `sns.countplot`, `sns.boxplot` |
| **b. Feature Selection** | Identify the most predictive features; remove redundant/correlated ones | Correlation matrix, `SelectKBest` with `f_classif`, or p-value analysis |
| **c. Train Model(s)** | Train and compare multiple classification algorithms | Logistic Regression, SVM, Naive Bayes (choose one or more) |
| **d. Interpret & Evaluate** | Compare models using classification reports; select the best and justify the choice | `classification_report`, confusion matrix, ROC-AUC |
| **e. Report** | Summarise findings for a non-technical audience in a PDF report | Matplotlib figures, written interpretation |

> **Note:** Replace the entries in this table with your actual plan. The table above is a template to guide you.

---
## 3. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif

# Classification algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# Load your dataset — update the filename to match yours
df = pd.read_csv('your_dataset.csv')
df.head()

---
## 4. Exploratory Data Analysis (EDA) & Data Cleaning

**Important:** Every visualisation you produce must be followed by a markdown cell interpreting what you see. Do not just generate graphs — tell the reader what the graph reveals about your data and how it influences your next steps.

### 4a. Basic Data Inspection

In [ ]:
# Shape and data types
print(f"Dataset shape: {df.shape}")
print("\nData Types:")
print(df.dtypes)

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

**Interpretation:** *(Write here — e.g., "The dataset contains X rows and Y features. There are no missing values, which confirms the dataset is clean. No duplicate rows were found, so no removal is necessary." OR describe the cleaning steps you took if issues were found.)*

### 4b. Class Distribution

Understanding class balance is critical before training any classification model.

In [ ]:
# Update 'diagnosis' to your actual target column name
target_col = 'diagnosis'

plt.figure(figsize=(6, 4))
sns.countplot(x=target_col, data=df, palette='Set2')
plt.title('Class Distribution (Target Variable)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(df[target_col].value_counts())
print(f"\nClass balance ratio: {df[target_col].value_counts(normalize=True).round(2).to_dict()}")

**Interpretation:** *(Write here — e.g., "The count plot shows that the dataset is moderately imbalanced, with X% Benign and Y% Malignant cases. This imbalance means accuracy alone will be a misleading metric — we will prioritise recall for the malignant class since false negatives carry a higher clinical risk. We will use a stratified train/test split to preserve this ratio.")*

### 4c. Feature Distributions (Histograms)

In [ ]:
# Plot histograms for all numeric features
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

df[numeric_cols].hist(figsize=(16, 12), bins=20, color='steelblue', edgecolor='white')
plt.suptitle('Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** *(Write here — e.g., "Several features such as X and Y appear right-skewed, suggesting outliers or a non-normal distribution. Features like Z appear approximately normally distributed. Skewed features may affect distance-based algorithms like SVM, which is why we will apply StandardScaler during preprocessing.")*

### 4d. Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    annot_kws={'size': 7}
)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

**Interpretation:** *(Write here — e.g., "The heatmap reveals that features A and B are highly correlated (r = 0.95), indicating multicollinearity. Retaining both would not add information to the model. We will address this during feature selection by identifying and removing redundant features. Features X and Y show moderate positive correlation with the target, suggesting they are strong predictors.")*

### 4e. Boxplots — Feature Distributions by Class

In [ ]:
# Select a subset of key features to visualise by class
# Update this list with your actual feature names
key_features = numeric_cols[:6]  # First 6 numeric features — adjust as needed

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feature in enumerate(key_features):
    sns.boxplot(x=target_col, y=feature, data=df, palette='Set2', ax=axes[i])
    axes[i].set_title(f'{feature} by Class')

plt.suptitle('Key Feature Distributions by Target Class', fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation:** *(Write here — e.g., "The boxplots show that feature X has a noticeably higher median value in malignant cases compared to benign cases, suggesting it has strong discriminating power. Feature Y shows significant overlap between classes, implying it may contribute less to the model. Outliers are visible in several features, which we will account for during scaling.")*

---
## 5. Feature Engineering & Selection

**Requirement:** Explain your strategy for encoding categorical features (if any) and selecting the most informative predictors.

In [ ]:
# --- Encode the target variable if it is categorical (e.g., 'M'/'B' or 'Yes'/'No') ---
le = LabelEncoder()
df[target_col] = le.fit_transform(df[target_col])
print(f"Class encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# --- Encode any other categorical columns using One-Hot Encoding ---
# df = pd.get_dummies(df, drop_first=True)  # Uncomment if you have categorical features

# --- Drop columns that are not useful (e.g., ID columns) ---
# df.drop(columns=['id'], inplace=True)  # Update with your actual column names

# --- Define X and y ---
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# --- Feature Selection using SelectKBest ---
# This ranks features by their statistical relationship with the target
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'F-Score': selector.scores_,
    'P-Value': selector.pvalues_
}).sort_values('F-Score', ascending=False)

print(feature_scores.to_string(index=False))

# Visualise top features
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_scores.head(15), x='F-Score', y='Feature', palette='viridis')
plt.title('Top Features by F-Score (SelectKBest)')
plt.tight_layout()
plt.show()

**Interpretation:** *(Write here — e.g., "The F-Score ranking confirms that features X, Y, and Z are the most statistically significant predictors of malignancy. Features with p-values above 0.05 (e.g., feature A) will be considered for removal as they show no statistically significant relationship with the target class. We will select the top K features for model training.")*

In [ ]:
# --- Select the top K features and apply scaling ---
# Update K based on your analysis above
top_k = 10
top_features = feature_scores.head(top_k)['Feature'].tolist()
X_selected = X[top_features]

# Scale features — critical for SVM and Logistic Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

# Stratified split to preserve class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

---
## 6. Model Training & Comparative Analysis

**Requirement:** Train at least one classification algorithm. To earn higher marks, train and compare multiple algorithms (e.g., Logistic Regression, SVM, Naive Bayes) and use the classification report to justify which model performs best.

In [ ]:
# --- Define models to compare ---
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes': GaussianNB()
}

# --- Train all models and store results ---
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'model': model,
        'predictions': preds,
        'report': classification_report(y_test, preds, output_dict=True)
    }
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, preds, target_names=le.classes_))

**Interpretation:** *(Write here — compare the classification reports. Which model achieved the highest precision, recall, and F1-score? Is there a trade-off between precision and recall? Which metric matters most for this problem and why?)*

### 6a. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(18, 5))

for ax, (name, result) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, result['predictions'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.suptitle('Confusion Matrices — Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation:** *(Write here — e.g., "The confusion matrix for SVM shows only 2 false negatives (malignant cases predicted as benign), which is lower than Logistic Regression's 4. In a cancer classification context, false negatives are particularly dangerous as they represent missed diagnoses. SVM therefore shows better clinical reliability for this task.")*

### 6b. ROC-AUC Comparison

In [ ]:
plt.figure(figsize=(8, 6))

for name, result in results.items():
    model = result['model']
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Model Comparison')
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation:** *(Write here — e.g., "The ROC curve shows that SVM achieves the highest AUC score of 0.98, meaning it is the best at distinguishing between malignant and benign tumours across all classification thresholds. A score closer to 1.0 indicates a near-perfect classifier. Naive Bayes, with an AUC of 0.93, still outperforms random classification but shows weaker discriminating ability.")*

---
## 7. Best Model Selection & Justification

**Requirement:** Based on your comparison, select one model as your final choice and justify it clearly.

In [ ]:
# --- Summary comparison table ---
summary_rows = []
for name, result in results.items():
    report = result['report']
    summary_rows.append({
        'Model': name,
        'Accuracy': round(report['accuracy'], 4),
        'Precision (Malignant)': round(report.get('1', report.get('Malignant', {})).get('precision', 0), 4),
        'Recall (Malignant)': round(report.get('1', report.get('Malignant', {})).get('recall', 0), 4),
        'F1-Score (Malignant)': round(report.get('1', report.get('Malignant', {})).get('f1-score', 0), 4),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

**Justification of Best Model:**

*(Write here. Address all of the following:)*
- *Which model did you select and why?*
- *Why is it better suited to this dataset than the others?*
- *Which metric drove your decision (accuracy, recall, F1, AUC) and why is that the right metric for this problem?*
- *What are the limitations of your chosen model?*

> *Example:* "SVM with an RBF kernel was selected as the final model. It achieved the highest recall for the malignant class (0.97), meaning it correctly identified 97% of actual cancer cases. In a clinical setting, missing a malignant case (false negative) is far more costly than a false alarm (false positive), making recall the priority metric. While Logistic Regression achieved similar accuracy, its lower recall on the malignant class makes it less suitable for this specific use case. The main limitation of SVM is that it is computationally expensive on large datasets and offers less interpretability than Logistic Regression."

---
## 8. Testing with Unseen (Synthetic) Data

**Context:** Some students raised concerns about testing on synthetic data. For this assessment, you are to use the real dataset for all training and evaluation. However, to demonstrate that your model generalises beyond the test set, you will generate a small number of synthetic but plausible data points and run them through your trained model.

These synthetic samples should be constructed based on real feature ranges from your EDA — not random noise.

In [ ]:
# --- Generate synthetic unseen samples ---
# Base these values on the feature ranges you observed during EDA
# The feature names below must match your top_features list

# Example: Replace with real feature names and plausible values
synthetic_data = pd.DataFrame([
    # Each dict = one synthetic patient
    # Values should reflect realistic ranges from your dataset
    {feat: df[feat].mean() * 1.2 for feat in top_features},  # High-value sample
    {feat: df[feat].mean() * 0.7 for feat in top_features},  # Low-value sample
    {feat: df[feat].mean() for feat in top_features},         # Average sample
])

# Scale using the same scaler fitted on training data
synthetic_scaled = scaler.transform(synthetic_data)

# Predict using the best model — update 'Support Vector Machine' to your chosen model
best_model_name = 'Support Vector Machine'  # Update this
best_model = results[best_model_name]['model']

synthetic_preds = best_model.predict(synthetic_scaled)
synthetic_labels = le.inverse_transform(synthetic_preds)

synthetic_data['Predicted Class'] = synthetic_labels
print("Synthetic Sample Predictions:")
print(synthetic_data[['Predicted Class']])

**Interpretation:** *(Write here — explain what each synthetic sample represents and whether the model's prediction makes sense given the feature values you used. E.g., "Sample 1 was constructed with feature values 20% above the dataset mean, which broadly reflects a more aggressive tumour profile. The model correctly classified it as Malignant, consistent with what we would expect from the EDA.")*

---
## 9. Conclusion

*(Write here — 1 to 2 paragraphs addressing:)*
- *What was the overall goal of the analysis?*
- *Which model performed best and what does that mean for the real-world problem?*
- *What are the limitations of this analysis and what would you improve with more time or data?*

---
## 10. References

Use Harvard referencing for all sources. This includes datasets, libraries, tutorials, and any AI tools used.

**Format:**
> Author(s) (Year) *Title*. Available at: URL (Accessed: Date).

**Example entries:**

Pedregosa, F. et al. (2011) *Scikit-learn: Machine Learning in Python*, Journal of Machine Learning Research, 12, pp. 2825–2830. Available at: https://scikit-learn.org

UCI Machine Learning Repository (1995) *Breast Cancer Wisconsin (Diagnostic) Dataset*. Available at: https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(diagnostic) (Accessed: [Your date]).

*(Add all additional references here.)*